# AI-Powered Indie & Mellow Music Engine — Skala Penuh (900K+ Lagu)
**Fokus Keahlian:** Data Science & Machine Learning — NLP Semantik (Sentence Embeddings) + Approximate Nearest Neighbor Search

**Latar Belakang Riset:**
Di era modern, musik telah bertransformasi menjadi rutinitas harian yang esensial. Bagi audiens kontemporer (khususnya Generasi Z), musisi *indie pop* dan balada naratif seperti Hindia, Bernadya, hingga Pamungkas telah menjadi primadona yang mendominasi preferensi musik. Aliran *indie pop* dan balada lokal akan menjadi fokus utama dalam riset ini.

Namun, di balik kemudahan algoritma *streaming* saat ini, sering kali muncul kendala teknis: *bagaimana jika fitur auto-playlist gagal menangkap nuansa spesifik yang kita inginkan?* Riset independen ini lahir dari kebutuhan untuk mengkurasi lagu-lagu berirama *mellow* dan akustik secara akurat, yang sering kali sulit dilakukan secara manual.

Sebagai solusi, prototipe mesin rekomendasi (*Recommendation Engine*) ini dibangun menggunakan hibrida *Machine Learning* dan *Deep Learning* untuk membedah kedalaman lirik dan fitur audio, menciptakan kurasi musik yang presisi tanpa bergantung pada algoritma bawaan platform.

**Spesifikasi Teknis & Pendekatan Model:**
1. **Skala Penuh (Full Scale):** Pemrosesan dilakukan pada seluruh ±900 ribu lagu di dalam dataset untuk memetakan variasi musik global secara komprehensif.
2. **Clustering Skalabel:** Penggunaan `MiniBatchKMeans` untuk menjaga efisiensi memori dan memangkas waktu komputasi pada dataset berskala raksasa.
3. **Deteksi Bahasa Berbasis AI:** Menggunakan model `fastText` terlatih untuk deteksi bahasa yang lebih akurat, jauh melampaui kemampuan heuristik daftar kata kunci konvensional.
4. **Deteksi Klaster Otomatis:** Sistem mendeteksi target klaster *mellow* secara dinamis dari nilai *centroid* (kombinasi *valence* dan *energy* terendah), menghasilkan alur kerja yang kokoh meski model dilatih ulang.
5. **Pencarian Semantik (FAISS):** Pemodelan *Sentence Embeddings* dipadukan dengan arsitektur FAISS (*Facebook AI Similarity Search*) untuk mengalkulasi kedekatan makna lirik secara kontekstual dengan latensi sangat rendah.

In [1]:
# Instalasi seluruh pustaka esensial (Pemrosesan Data, Machine Learning, NLP, dan FAISS)
!pip install -q kagglehub git+https://github.com/facebookresearch/fastText.git sentence-transformers faiss-cpu pyarrow "numpy<2.0"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import pandas as pd
import os
import kagglehub

# 1. Unduh & muat SELURUH dataset Spotify (tanpa sampling)
path = kagglehub.dataset_download("bwandowando/spotify-songs-with-attributes-and-lyrics")
print(f"Dataset tersimpan di: {path}")

print("Sedang memuat dataset spotify data asli...")
df_mentah = pd.read_csv(os.path.join(path, "songs_with_attributes_and_lyrics.csv"))

# 2. Menginspeksi wujud asli data sebelum direduksi
print("\n--- ARSITEKTUR DATA MENTAH ---")
print(f"Dimensi (Baris, Kolom): {df_mentah.shape}")
print("\nDaftar Kolom Asli:")
for i, kolom in enumerate(df_mentah.columns, 1):
    print(f"{i}. {kolom}")

Using Colab cache for faster access to the 'spotify-songs-with-attributes-and-lyrics' dataset.
Dataset tersimpan di: /kaggle/input/spotify-songs-with-attributes-and-lyrics
Sedang memuat dataset spotify data asli...

--- ARSITEKTUR DATA MENTAH ---
Dimensi (Baris, Kolom): (955320, 17)

Daftar Kolom Asli:
1. id
2. name
3. album_name
4. artists
5. danceability
6. energy
7. key
8. loudness
9. mode
10. speechiness
11. acousticness
12. instrumentalness
13. liveness
14. valence
15. tempo
16. duration_ms
17. lyrics


### Kamus Data (Data Dictionary) Spotify API
Dataset ini terdiri dari 17 variabel yang terbagi menjadi tiga kategori utama: Metadata Identitas, Fitur Analisis Audio (*Audio Features*), dan Data Tekstual.

| Kategori | Nama Kolom | Deskripsi |
| :--- | :--- | :--- |
| **Metadata** | `id` | ID alfanumerik unik Spotify untuk trek lagu. |
| | `name` | Judul lagu. |
| | `album_name` | Nama album dari lagu tersebut. |
| | `artists` | Nama musisi atau grup band. |
| | `duration_ms` | Durasi trek dalam satuan milidetik. |
| **Data Teks** | `lyrics` | Teks lirik lagu utuh hasil ekstraksi. |
| **Fitur Audio** | `valence` | Skor (0.0 - 1.0) tingkat emosi positif (ceria/bahagia). Semakin mendekati 0, semakin *mellow*/sedih. |
| | `acousticness` | Skor (0.0 - 1.0) tingkat keyakinan bahwa trek tersebut murni instrumen akustik tanpa *synthesizer*. |
| | `energy` | Skor (0.0 - 1.0) intensitas audio. Energi rendah berarti lagu bertempo lambat dan tenang. |
| | `danceability` | Skor (0.0 - 1.0) kecocokan irama dan ketukan lagu untuk digunakan menari. |
| | `instrumentalness` | Prediksi (0.0 - 1.0) ketiadaan vokal. Skor tinggi berarti lagu murni instrumen. |
| | `speechiness` | Deteksi vokal lisan (seperti *podcast*, *rap*, atau *spoken word*). |
| | `liveness` | Deteksi apakah lagu direkam secara *live* bersama penonton atau di dalam studio. |
| | `tempo` | Kecepatan trek dalam ukuran *Beats Per Minute* (BPM). |
| | `loudness` | Rata-rata tingkat volume trek dalam desibel (dB). |
| | `key` | Kunci nada dasar trek (menggunakan notasi standar *Pitch Class*). |
| | `mode` | Modalitas melodi trek (Mayor = 1, Minor = 0). |

*Catatan: Pemahaman mendalam terhadap kamus data ini menjadi landasan logis bagi proses Reduksi Dimensi dan Seleksi Fitur pada tahap selanjutnya.*

### Seleksi Fitur (Feature Selection) & Kutukan Dimensi
Kita menyeleksi empat metrik audio utama dan membuang fitur lainnya untuk menghindari *Curse of Dimensionality* (dimensi berlebih yang dapat menambahkan *noise* dan mengurangi performa model):

*   **`valence` (Skor Emosi):** Rentang 0.0 hingga 1.0. Semakin mendekati 1.0, nuansa lagu semakin positif (ceria/bahagia); semakin mendekati 0.0, nuansa lagu semakin negatif (sedih/mellow).
*   **`acousticness` (Skor Akustik):** Rentang 0.0 hingga 1.0. Semakin tinggi skornya, semakin dominan instrumen akustiknya dibandingkan suara elektronik atau *synthesizer*.
*   **`energy` (Intensitas Audio):** Rentang 0.0 hingga 1.0. Skor tinggi mengindikasikan lagu bertempo cepat, bising, atau intens (seperti genre *rock* atau *metal*).
*   **`danceability` (Skor Dansa):** Rentang 0.0 hingga 1.0. Menunjukkan seberapa cocok ritme dan ketukan lagu untuk digunakan menari.

**Catatan Pendekatan:** Pemrosesan dilakukan pada *seluruh* baris data yang memiliki lirik valid, memaksimalkan potensi ekstraksi data bahasa alami (NLP).

In [3]:
# 1. Reduksi 8 kolom target & buang baris tanpa lirik (SELURUH data, tanpa sampling)
kolom_target = ['id', 'name', 'artists', 'lyrics', 'valence', 'acousticness', 'energy', 'danceability']
df_valid = df_mentah[kolom_target].dropna(subset=['lyrics']).copy()
df_valid = df_valid[df_valid['lyrics'].astype(str).str.strip().str.len() > 0]
df_valid.reset_index(drop=True, inplace=True)

print(f"Total lagu mentah          : {df_mentah.shape[0]:,}")
print(f"Total lagu dengan lirik valid (diproses semua, tanpa sampling): {df_valid.shape[0]:,}")
display(df_valid[['name', 'artists']].head(3))

Total lagu mentah          : 955,320
Total lagu dengan lirik valid (diproses semua, tanpa sampling): 955,307


,name,artists
0,!,HELLYEAH
1,!!,Yxngxr1
2,!!! - Interlude,Glowie


In [4]:
# 1. Mengaudit tipe data dan memastikan struktur bersih pada SELURUH data
print("--- PROFIL STRUKTUR DATA (SELURUH DATASET) ---")
df_valid.info()

# 2. Membedah distribusi statistik 4 metrik esensial
print("\n--- DISTRIBUSI STATISTIK FITUR AUDIO ---")
fitur_audio_stats = df_valid[['valence', 'acousticness', 'energy', 'danceability']]
display(fitur_audio_stats.describe())

--- PROFIL STRUKTUR DATA (SELURUH DATASET) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 955307 entries, 0 to 955306
Data columns (total 8 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   id            955307 non-null  object 
 1   name          955296 non-null  object 
 2   artists       955302 non-null  object 
 3   lyrics        955307 non-null  object 
 4   valence       955307 non-null  float64
 5   acousticness  955307 non-null  float64
 6   energy        955307 non-null  float64
 7   danceability  955307 non-null  float64
dtypes: float64(4), object(4)
memory usage: 58.3+ MB

--- DISTRIBUSI STATISTIK FITUR AUDIO ---


,valence,acousticness,energy,danceability
count,955307.000000,955307.000000,955307.000000,955307.000000
mean,0.488119,0.282963,0.652441,0.550710
std,0.251467,0.311801,0.238823,0.169784
min,0.000000,0.000000,0.000000,0.000000
25%,0.282000,0.011900,0.482000,0.436000
50%,0.477000,0.142000,0.687000,0.558000
75%,0.690000,0.518000,0.857000,0.675000
max,1.000000,0.996000,1.000000,0.993000


## Unsupervised Learning — MiniBatchKMeans (Skalabel)
Karena dataset lagu ini tidak memiliki label kategori emosi yang eksplisit (seperti "Galau", "Ceria", atau "Bising"), kita menggunakan pendekatan *Unsupervised Machine Learning*.

Algoritma pengelompokan (*Clustering*) akan mengukur jarak matematis dari keempat pilar metrik audio (*valence, acousticness, energy, danceability*). Secara intuisi, dunia musik secara garis besar dapat dikelompokkan ke dalam beberapa spektrum emosi utama (misalnya: *Rock/Bising, EDM/Dansa, Pop/Galau, Akustik Ceria,* dan *Balada/Mellow*).

**Mengapa menggunakan `MiniBatchKMeans`?**
Algoritma `KMeans` klasik menghitung ulang jarak seluruh titik data di setiap iterasi, yang sangat membebani memori pada ratusan ribu baris data. `MiniBatchKMeans` menyelesaikan masalah ini dengan mempelajari potongan kecil data (*mini-batch*) secara bertahap, memberikan hasil yang nyaris identik namun dengan waktu komputasi yang jauh lebih efisien.

### Standarisasi Skala (*Feature Scaling*)
Sebelum model dilatih, *Feature Scaling* wajib dilakukan. Algoritma berbasis jarak sangat sensitif terhadap rentang skala metrik. Menggunakan `StandardScaler` akan menormalisasi seluruh metrik sehingga memiliki bobot rata-rata 0 dan varians 1, mencegah fitur dengan nilai metrik besar mendominasi pembentukan klaster.

In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import MiniBatchKMeans

# 1. Standarisasi Skala Fitur (Feature Scaling) pada SELURUH dataset
fitur_audio = df_valid[['valence', 'acousticness', 'energy', 'danceability']]
scaler = StandardScaler()
audio_scaled = scaler.fit_transform(fitur_audio)

print(f"Melatih MiniBatchKMeans pada {audio_scaled.shape[0]:,} lagu...")

# 2. Pelatihan Model Clustering
kmeans = MiniBatchKMeans(
    n_clusters=5,          # Mengelompokkan data ke dalam 5 nuansa utama
    random_state=42,       # Memastikan hasil klaster dapat direproduksi secara konsisten
    n_init=10,             # Melakukan 10 kali inisialisasi centroid awal untuk mencari yang paling optimal
    batch_size=10_000,     # Memproses data per 10.000 baris untuk efisiensi memori
)

# Mempelajari distribusi data sekaligus menghasilkan prediksi label klaster
df_valid['klaster_vibes'] = kmeans.fit_predict(audio_scaled)

# 3. Analisis Nilai Tengah (Centroid Analysis)
print("\nKarakteristik Rata-Rata Metrik untuk ke-5 Klaster:")
hasil_klaster = df_valid.groupby('klaster_vibes')[['valence', 'acousticness', 'energy', 'danceability']].mean()
display(hasil_klaster)

Melatih MiniBatchKMeans pada 955,307 lagu...

Karakteristik Rata-Rata Metrik untuk ke-5 Klaster:


,valence,acousticness,energy,danceability
klaster_vibes,,,,
0,0.291033,0.047301,0.837325,0.362884
1,0.550478,0.640286,0.450735,0.619676
2,0.802329,0.202690,0.734743,0.707891
3,0.474923,0.093393,0.739786,0.596282
4,0.234911,0.738580,0.295141,0.413580


### Membaca Karakteristik Klaster
Tabel hasil *centroid* di atas berfungsi sebagai "kompas" untuk mengidentifikasi klaster mana yang merepresentasikan lagu *indie/mellow*. Pola interpretasinya adalah sebagai berikut:
* **Valence & Energy Rendah** → Lagu bergenre balada/mellow, bersuasana syahdu, sering kali didominasi instrumen akustik. **(Ini adalah target utama kita).**
* **Energy Tinggi, Acousticness Rendah** → Lagu bising dan intens (seperti *Rock, Metal, Pop-Punk*).
* **Valence & Danceability Tinggi** → Lagu pop riang dengan tempo cepat atau musik *EDM*.
* **Valence Rendah, Energy Sedang-Tinggi** → Lagu bernuansa sedih namun diiringi oleh instrumen *full band* yang cukup kuat.

**Catatan Penting:** ID klaster (0–4) bersifat dinamis dan dapat berubah bergantung pada inisialisasi awal (*random seed*). Oleh karena itu, logika pencarian klaster target pada tahapan selanjutnya dilakukan secara dinamis (otomatis) berdasarkan nilai minimum dari parameter *valence* dan *energy*, bukan dengan menebak ID klaster secara manual (misalnya `klaster_vibes == 1`).

In [6]:
# Ekstraksi otomatis klaster Mellow/Balada.
# Karakteristik utama: tingkat kebahagiaan (valence) dan intensitas audio (energy) yang sangat rendah.
# Kita menjumlahkan rata-rata valence dan energy tiap klaster, lalu mencari nilai minimumnya.
skor_mellow = hasil_klaster['valence'] + hasil_klaster['energy']
klaster_mellow_id = skor_mellow.idxmin()

print(f"Klaster bernuansa 'mellow' berhasil dideteksi otomatis pada: Klaster {klaster_mellow_id}")
display(hasil_klaster.loc[[klaster_mellow_id]])

# Memisahkan lagu-lagu di dalam klaster target ke DataFrame baru
df_mellow = df_valid[df_valid['klaster_vibes'] == klaster_mellow_id].copy()
df_mellow.reset_index(drop=True, inplace=True)
print(f"\nTotal lagu pada klaster mellow: {df_mellow.shape[0]:,} lagu (dari total keseluruhan {df_valid.shape[0]:,} lagu).")

Klaster bernuansa 'mellow' berhasil dideteksi otomatis pada: Klaster 4


,valence,acousticness,energy,danceability
klaster_vibes,,,,
4,0.234911,0.73858,0.295141,0.41358



Total lagu pada klaster mellow: 131,302 lagu (dari total keseluruhan 955,307 lagu).


## Ekstraksi Bahasa Lokal & Prapemrosesan NLP
Dataset mentah Spotify memuat hampir sejuta lagu dari seluruh dunia. Karena target mesin rekomendasi ini adalah musik *indie* dan balada naratif Indonesia, dataset akan difilter menggunakan **fastText** (`lid.176.bin`) — model identifikasi bahasa terlatih pada 176 bahasa yang mampu memproses teks dalam waktu singkat dengan tingkat akurasi tinggi.

**Optimasi Alur Komputasi:** Deteksi bahasa dan sterilisasi teks adalah proses operasional per-baris yang memakan sumber daya besar. Oleh karena itu, tahapan NLP ini secara spesifik dieksekusi **hanya setelah** klaster *mellow* berhasil diisolasi dari keseluruhan 900K+ data. Hal ini memangkas beban komputasi hingga 85% tanpa membuang satupun lagu target. Selanjutnya, lirik disterilisasi dari tanda baca dan penanda struktur lagu (seperti `[Chorus]`) agar model berfokus mutlak pada representasi semantik kata.

In [7]:
import re
import os
import fasttext

# 1. Pemuatan model identifikasi bahasa fastText
if not os.path.exists('lid.176.bin'):
    !wget -q https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin
model_lang = fasttext.load_model('lid.176.bin')

def deteksi_bahasa_robust(teks):
    t_bersih = str(teks).lower()
    # Buang header/metadata umum ala Genius sebelum diproses
    t_bersih = re.sub(r'^\d+\s*contributors?.*?lyrics', ' ', t_bersih, flags=re.DOTALL)
    t_bersih = re.sub(r'translations.*?(?=\n|\[)', ' ', t_bersih, flags=re.DOTALL)
    t_bersih = re.sub(r'\[.*?\]|\(.*?\)', ' ', t_bersih)
    t_bersih = re.sub(r'[^a-z\s]', ' ', t_bersih)
    t_bersih = ' '.join(t_bersih.split())

    # Ambil jendela teks dari TENGAH lirik, bukan cuma awal
    n = len(t_bersih)
    if n < 15:
        return 'unknown'
    start = max(0, (n // 2) - 150)
    t_input = t_bersih[start:start + 300]

    try:
        labels, probs = model_lang.predict(t_input, k=1)
        pred_label = str(labels[0][0]).lower()
        conf = probs[0]
        if ('id' in pred_label or 'ind' in pred_label or 'ms' in pred_label) and conf > 0.5:
            return 'id'
    except:
        pass

    # Fallback tetap dicek meski fastText "yakin" salah — bukan cuma saat error
    kata_kunci_indo = {'yang','aku','kamu','dan','dengan','bisa','tidak','kau','akan','untuk','mereka','tak','ku','mu','ini','itu','kita'}
    kata_lirik = set(t_bersih.split()[:80])
    if len(kata_lirik.intersection(kata_kunci_indo)) >= 3:
        return 'id'

    return 'other'

def bersihkan_lirik(teks):
    # Membersihkan lirik dari karakter spesial untuk persiapan Sentence Embeddings
    t = str(teks).lower()
    t = re.sub(r'\[.*?\]|\(.*?\)', ' ', t)
    t = re.sub(r'[^a-zàâçéèêëîïôûùüÿñæœ\s]', ' ', t)
    return ' '.join(t.split())

# Menggabungkan klaster yang memiliki kedekatan nuansa (misal: Mellow Pop dan Balada Akustik)
klaster_target = [1, 4]
print(f"Mengonsolidasi lagu dari Klaster {klaster_target}...")
df_mellow_gabungan = df_valid[df_valid['klaster_vibes'].isin(klaster_target)].copy()
df_mellow_gabungan.reset_index(drop=True, inplace=True)

print(f"Mengeksekusi deteksi bahasa pada {df_mellow_gabungan.shape[0]:,} lagu...")
df_mellow_gabungan['bahasa'] = df_mellow_gabungan['lyrics'].apply(deteksi_bahasa_robust)

# Filtrasi akhir khusus lagu berbahasa Indonesia
df_indo = df_mellow_gabungan[df_mellow_gabungan['bahasa'] == 'id'].copy()
df_indo['lirik_clean'] = df_indo['lyrics'].apply(bersihkan_lirik)
df_indo['lirik_clean'] = df_indo['lirik_clean'].fillna('')
df_indo = df_indo[df_indo['lirik_clean'].str.len() > 20]
df_indo['lirik_bersih'] = df_indo['lirik_clean']
df_indo.reset_index(drop=True, inplace=True)

print(f"\nProses Selesai: {df_indo.shape[0]:,} lagu Indonesia bernuansa mellow berhasil dikurasi.")
if df_indo.shape[0] > 0:
    display(df_indo[['name', 'artists', 'klaster_vibes']].head(15))


Mengonsolidasi lagu dari Klaster [1, 4]...
Mengeksekusi deteksi bahasa pada 281,947 lagu...

Proses Selesai: 538 lagu Indonesia bernuansa mellow berhasil dikurasi.


,name,artists,klaster_vibes
0,'Ego',Ekamatra,1
1,1904.,YAPH,1
2,1st Battalion Bugle Call / Fall In,['The Corps of Drums of the 1st Battalion The ...,1
3,8 Tahun,Adhitia Sofyan,4
4,A,Altimet,1
5,"A Christmas Carol - Stave Three, The Second of...",Bart Wolffe,1
6,Aduh! Usahlah Berubah,Samudera,4
7,Aduhai ! Seribu Kali Sayang,Iklim,4
8,Ajarkan Aku...,Arvian Dwi,4
9,Akhir Cerita Cinta,Glenn Fredly,4


In [12]:
# Diagnostic run: deteksi bahasa di SELURUH df_valid, tanpa filter klaster dulu
df_valid['bahasa'] = df_valid['lyrics'].apply(deteksi_bahasa_robust)
df_id_semua = df_valid[df_valid['bahasa'] == 'id']

print(f"Total lagu Indonesia di SELURUH dataset: {df_id_semua.shape[0]:,}")
print("\nSebaran klaster untuk lagu-lagu Indonesia ini:")
display(df_id_semua['klaster_vibes'].value_counts())

Total lagu Indonesia di SELURUH dataset: 1,508

Sebaran klaster untuk lagu-lagu Indonesia ini:


,count
klaster_vibes,
3,444
2,276
4,274
1,264
0,250


## Mesin Rekomendasi NLP — Sentence Embeddings + FAISS

**Mengapa Meninggalkan Pendekatan Klasik?** Pendekatan tradisional seperti TF-IDF dan *Cosine Similarity* mengukur kemiripan **secara leksikal (kecocokan kata harfiah)**. Dua lirik dengan makna serupa namun menggunakan diksi berbeda ("hujan membasahi kenangan" vs "gerimis mengiringi rindu") akan dinilai tidak relevan. Metode ini juga menghasilkan matriks *sparse* yang luar biasa besar dan gagal menangkap sinonim, konteks, serta kedalaman emosional teks.

**Arsitektur Modern (Sentence Embeddings):** Menggunakan model *Transformer* multibahasa (`paraphrase-multilingual-MiniLM-L12-v2`), setiap teks lirik dikonversi menjadi vektor numerik padat (*dense vector*) yang merepresentasikan **makna semantik** keseluruhan kalimat. Dua lagu bernuansa sedih yang menggunakan perumpamaan berbeda akan memiliki koordinat vektor yang saling berdekatan di dalam ruang multidimensi.

**Akselerasi Pencarian (FAISS):** Menghitung *Cosine Similarity* secara penuh untuk ratusan ribu lagu akan melumpuhkan memori komputasi standar. Dengan mengimplementasikan FAISS (*Facebook AI Similarity Search*), sistem membangun hierarki *index vektor* pintar. Hal ini memungkinkan sistem untuk mencari tetangga-terdekat (*K-Nearest Neighbors*) dalam hitungan milidetik, mengadopsi standar sistem rekomendasi skala industri yang sesungguhnya.

In [8]:
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

# 1. Model embedding multibahasa yang ringan & mendukung Bahasa Indonesia secara native
model_embed = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# 2. Encode SELURUH lirik hasil filter menjadi vektor semantik (batch agar hemat memori)
print(f"Membuat embedding semantik untuk {df_indo.shape[0]:,} lirik...")
embeddings = model_embed.encode(
    df_indo['lirik_bersih'].tolist(),
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
)

# 3. Normalisasi vektor -> inner product di FAISS jadi setara cosine similarity
faiss.normalize_L2(embeddings)

# 4. Bangun index FAISS untuk pencarian tetangga terdekat yang cepat & skalabel
dimensi = embeddings.shape[1]
index_faiss = faiss.IndexFlatIP(dimensi)
index_faiss.add(embeddings)

print(f"Index FAISS siap: {index_faiss.ntotal:,} lagu ter-index, dimensi vektor = {dimensi}.")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Membuat embedding semantik untuk 538 lirik...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Index FAISS siap: 538 lagu ter-index, dimensi vektor = 384.


In [9]:
def rekomendasikan_lagu_indo(index, jumlah=3):
    # Mengambil vektor semantik dari lagu referensi
    query_vec = embeddings[index:index + 1]

    # Melakukan pencarian tetangga terdekat di FAISS
    # Menambahkan +1 karena hasil teratas selalu merupakan lagu itu sendiri
    skor, idx_tetangga = index_faiss.search(query_vec, jumlah + 1)

    judul = df_indo.iloc[index]['name']
    artis = df_indo.iloc[index]['artists']
    print(f"Referensi Pencarian: '{judul}' oleh {artis}\n")
    print("Sistem merekomendasikan lagu-lagu berikut berdasarkan kemiripan makna lirik:")

    # Menyaring hasil agar tidak menampilkan lagu referensi
    hasil = [(idx, s) for idx, s in zip(idx_tetangga[0], skor[0]) if idx != index][:jumlah]
    for i, (idx, s) in enumerate(hasil, 1):
        j = df_indo.iloc[idx]['name']
        a = df_indo.iloc[idx]['artists']
        print(f"{i}. {j} - {a} (Skor Kesamaan Semantik: {s:.2f})")

# -- Pengujian Modul Rekomendasi --
print("-" * 50)
target_musisi = "Yura Yunita"

# Pencarian dinamis indeks lagu berdasarkan nama musisi
pencarian = df_indo[df_indo['artists'].str.contains(target_musisi, case=False, na=False)]

if not pencarian.empty:
    index_ditemukan = pencarian.index[0]
    rekomendasikan_lagu_indo(index=index_ditemukan)
else:
    print(f"Musisi '{target_musisi}' tidak ditemukan di dalam dataset akhir berbahasa Indonesia.")
    display(df_indo[['name', 'artists']].head(20))

--------------------------------------------------
Referensi Pencarian: 'Berawal Dari Tatap' oleh Yura Yunita

Sistem merekomendasikan lagu-lagu berikut berdasarkan kemiripan makna lirik:
1. Sosok Sempurna - Stand Here Alone (Skor Kesamaan Semantik: 0.82)
2. Kerana Pengalaman - Zaleha Hamid (Skor Kesamaan Semantik: 0.81)
3. Terlena - Ikke Nurjanah (Skor Kesamaan Semantik: 0.81)


## Menyimpan Artefak Model
Membangun embedding untuk ratusan ribu lirik butuh waktu — simpan hasilnya agar tidak perlu mengulang seluruh pipeline dari awal setiap kali notebook dibuka ulang.

In [10]:
import pickle

faiss.write_index(index_faiss, 'index_lagu_mellow.faiss')
df_indo.to_parquet('df_lagu_mellow_indo.parquet', index=False)
np.save('embeddings_lagu_mellow.npy', embeddings)

print("Tersimpan: index_lagu_mellow.faiss, df_lagu_mellow_indo.parquet, embeddings_lagu_mellow.npy")
print("Lain kali cukup load ketiga file ini via faiss.read_index() / pd.read_parquet() / np.load(),")
print("tanpa perlu mengulang clustering, deteksi bahasa, atau encoding embedding dari nol.")

Tersimpan: index_lagu_mellow.faiss, df_lagu_mellow_indo.parquet, embeddings_lagu_mellow.npy
Lain kali cukup load ketiga file ini via faiss.read_index() / pd.read_parquet() / np.load(),
tanpa perlu mengulang clustering, deteksi bahasa, atau encoding embedding dari nol.
